# Touche23-ValueEval Preprocessing

Transforms the Touche23 TSV files into `data/touche_positive_only.csv` with columns:
`argument_id, question, value, positive_answer, negative_answer`

Multi-label arguments are **duplicated** — one row per active value label.
The `negative_answer` column is left empty for the LLM pipeline to fill.

In [4]:
from pathlib import Path
import pandas as pd

In [5]:
DATA_DIR = Path("data")

SPLITS = ["training", "validation", "test"]

VALUE_COLUMNS = [
    "Self-direction: thought",
    "Self-direction: action",
    "Stimulation",
    "Hedonism",
    "Achievement",
    "Power: dominance",
    "Power: resources",
    "Face",
    "Security: personal",
    "Security: societal",
    "Tradition",
    "Conformity: rules",
    "Conformity: interpersonal",
    "Humility",
    "Benevolence: caring",
    "Benevolence: dependability",
    "Universalism: concern",
    "Universalism: nature",
    "Universalism: tolerance",
    "Universalism: objectivity",
]

In [3]:
def conclusion_to_question(conclusion: str) -> str:
    """
    Convert a Touche conclusion statement into a question.

    Handles three patterns found in the corpus:
      1. "We should X"           → "Should we X?"
      2. "X should Y"            → "Should X Y?"
      3. Declarative (no should) → append "?"
    """
    c = conclusion.strip().rstrip(".")
    lower = c.lower()

    if lower.startswith("we should "):
        rest = c[len("we should "):].strip()
        return f"Should we {rest}?"

    if " should " in lower:
        idx = lower.index(" should ")
        subject = c[:idx].strip()
        predicate = c[idx + len(" should "):].strip()
        return f"Should {subject} {predicate}?"

    # Declarative conclusions (e.g., "X brings more harm than good")
    return f"{c}?"

In [4]:
# Smoke-test the converter on known patterns
examples = [
    "We should ban human cloning",
    "We should end the use of economic sanctions",
    "5G services should be available across India",
    "Beauty pageants should be banned",
    "Social media brings more harm than good",
    "Homeopathy brings more harm than good.",
]
for ex in examples:
    print(f"  {ex!r}")
    print(f"→ {conclusion_to_question(ex)!r}")
    print()

  'We should ban human cloning'
→ 'Should we ban human cloning?'

  'We should end the use of economic sanctions'
→ 'Should we end the use of economic sanctions?'

  '5G services should be available across India'
→ 'Should 5G services be available across India?'

  'Beauty pageants should be banned'
→ 'Should Beauty pageants be banned?'

  'Social media brings more harm than good'
→ 'Social media brings more harm than good?'

  'Homeopathy brings more harm than good.'
→ 'Homeopathy brings more harm than good?'



In [5]:
def load_split(split: str) -> pd.DataFrame:
    """
    Load one arguments + labels split, join them, melt the value columns,
    and keep only rows where the label == 1.
    Returns a DataFrame with columns:
      argument_id, split, question, stance, value, positive_answer
    """
    args_path   = DATA_DIR / f"arguments-{split}.tsv"
    labels_path = DATA_DIR / f"labels-{split}.tsv"

    args_df   = pd.read_csv(args_path,   sep="\t")
    labels_df = pd.read_csv(labels_path, sep="\t")

    # Normalise column name used as join key
    args_df   = args_df.rename(columns={"Argument ID": "argument_id"})
    labels_df = labels_df.rename(columns={"Argument ID": "argument_id"})

    merged = args_df.merge(labels_df, on="argument_id")

    # Melt the 20 value columns → one row per (argument, value) pair
    melted = merged.melt(
        id_vars=["argument_id", "Conclusion", "Stance", "Premise"],
        value_vars=VALUE_COLUMNS,
        var_name="value",
        value_name="label",
    )

    # Keep only active labels
    melted = melted[melted["label"] == 1].copy()

    melted["split"]           = split
    melted["question"]        = melted["Conclusion"].apply(conclusion_to_question)
    melted["positive_answer"] = melted["Premise"]

    return melted[["argument_id", "split", "question", "Stance", "value", "positive_answer"]].rename(
        columns={"Stance": "stance"}
    )

In [6]:
frames = [load_split(s) for s in SPLITS]

for split, frame in zip(SPLITS, frames):
    print(f"{split:12s}: {len(frame):>6,} rows  (from {frame['argument_id'].nunique():>5,} unique arguments)")

df = pd.concat(frames, ignore_index=True)
print(f"{'TOTAL':12s}: {len(df):>6,} rows  (from {df['argument_id'].nunique():>5,} unique arguments)")

training    : 18,370 rows  (from 5,392 unique arguments)
validation  :  6,359 rows  (from 1,896 unique arguments)
test        :  4,771 rows  (from 1,576 unique arguments)
TOTAL       : 29,500 rows  (from 8,864 unique arguments)


In [7]:
print("Value distribution across all rows:")
print(df["value"].value_counts().to_string())

Value distribution across all rows:
value
Universalism: concern         3356
Security: personal            3296
Security: societal            2613
Achievement                   2499
Benevolence: caring           2301
Self-direction: action        2282
Conformity: rules             1919
Universalism: objectivity     1896
Self-direction: thought       1382
Benevolence: dependability    1237
Universalism: tolerance       1082
Tradition                      908
Power: dominance               882
Power: resources               862
Universalism: nature           698
Face                           608
Humility                       596
Stimulation                    462
Conformity: interpersonal      320
Hedonism                       301


In [8]:
print("Question-conversion pattern coverage:")
def classify_question(q):
    if q.lower().startswith("should we "):
        return "should-we"
    if q.lower().startswith("should "):
        return "should-subject"
    return "declarative"

df["q_type"] = df["question"].apply(classify_question)
print(df["q_type"].value_counts().to_string())
df = df.drop(columns=["q_type"])

Question-conversion pattern coverage:
q_type
should-we         21995
should-subject     4915
declarative        2590


In [9]:
# Inspect a few duplicated multi-label examples
multi_label_ids = (
    df.groupby("argument_id")["value"]
    .count()
    .loc[lambda s: s > 1]
    .index
)
print(f"Arguments with >1 active value: {len(multi_label_ids):,}\n")

sample_id = multi_label_ids[0]
print(f"Example duplicated argument ({sample_id}):")
display(df[df["argument_id"] == sample_id][["argument_id", "value", "question", "positive_answer"]])

Arguments with >1 active value: 8,116

Example duplicated argument (A01006):


,argument_id,value,question,positive_answer
4314,A01006,Power: dominance,Should we end the use of economic sanctions?,sometimes economic sanctions are the only thin...
7932,A01006,Security: societal,Should we end the use of economic sanctions?,sometimes economic sanctions are the only thin...


In [ ]:
# Add empty negative_answer column and save
df["negative_answer"] = ""

output_cols = ["argument_id", "split", "stance", "question", "value", "positive_answer", "negative_answer"]
df = df[output_cols].reset_index(drop=True)

out_path = DATA_DIR / "touche_positive_only.csv"
df.to_csv(out_path, index=False)
print(f"Saved {len(df):,} rows → {out_path}")

NameError: name 'df' is not defined

In [6]:
# Sanity check: reload and preview
out_path = DATA_DIR / "touche_positive_only.csv"

check = pd.read_csv(out_path)
print(f"Reloaded: {len(check):,} rows, columns: {list(check.columns)}")
check.head(5)

Reloaded: 29,500 rows, columns: ['argument_id', 'split', 'stance', 'question', 'value', 'positive_answer', 'negative_answer']


,argument_id,split,stance,question,value,positive_answer,negative_answer
0,A01010,training,against,Should we prohibit school prayer?,Self-direction: thought,it should be allowed if the student wants to p...,NaN
1,A01020,training,in favor of,Should we subsidize journalism?,Self-direction: thought,It is important for news organizations to tran...,NaN
2,A02015,training,in favor of,Should we ban the Church of Scientology?,Self-direction: thought,they are nothing but a cult and manipulate the...,NaN
3,A02020,training,against,Should we subsidize Wikipedia?,Self-direction: thought,wikipedia can form biased opinions on topics o...,NaN
4,A04003,training,in favor of,Should we abandon the use of school uniform?,Self-direction: thought,School uniforms keep children from being able ...,NaN


In [7]:
print("\\n" + "="*80)
print("VALUE FREQUENCY ANALYSIS")
print("="*80)
value_counts = check["value"].value_counts().sort_values(ascending=False)
print(f"\\nTotal rows: {len(check):,}")
print(f"Unique values: {check['value'].nunique()}")
print(f"\\nFrequency of each value:")
print()
for value, count in value_counts.items():
    pct = 100 * count / len(check)
    print(f"  {value:40s}  {count:>6,}  ({pct:5.1f}%)")
print()
print("Summary statistics:")
print(f"  Mean rows per value: {value_counts.mean():.0f}")
print(f"  Median rows per value: {value_counts.median():.0f}")
print(f"  Std deviation: {value_counts.std():.0f}")

\n================================================================================
VALUE FREQUENCY ANALYSIS
\nTotal rows: 29,500
Unique values: 20
\nFrequency of each value:

  Universalism: concern                      3,356  ( 11.4%)
  Security: personal                         3,296  ( 11.2%)
  Security: societal                         2,613  (  8.9%)
  Achievement                                2,499  (  8.5%)
  Benevolence: caring                        2,301  (  7.8%)
  Self-direction: action                     2,282  (  7.7%)
  Conformity: rules                          1,919  (  6.5%)
  Universalism: objectivity                  1,896  (  6.4%)
  Self-direction: thought                    1,382  (  4.7%)
  Benevolence: dependability                 1,237  (  4.2%)
  Universalism: tolerance                    1,082  (  3.7%)
  Tradition                                    908  (  3.1%)
  Power: dominance                             882  (  3.0%)
  Power: resources              